In [ ]:
from libs import circuits, dataloading, qnn_models
from libs.helpers import mix_with_white
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
from joblib import cpu_count, Parallel, delayed
from pathlib import Path
import glob
import numpy as np
import pickle
import json
from copy import deepcopy
import jax
import pennylane as qml
import csv
import json
import os
import pandas as pd
import keras
from keras import Sequential
import tensorflow as tf
import seaborn as sns
import time
import re
from cmcrameri import cm
from sklearn.metrics import explained_variance_score, r2_score, mean_squared_error
from collections import defaultdict

plt.style.use('./libs/notebookstyle_aps.mplstyle')

%load_ext autoreload
%autoreload 2

## overview of the PALM experiments

In [ ]:
datasets = pd.read_csv('/home/b/b309252/turbulence-qml/QNN_regression/dataset/PALMruns_overview.txt')

In [ ]:
mycolors = cm.batlowS(np.linspace(0,1,10)[3:9])

plt.scatter(datasets.iloc[:18].u, datasets.iloc[:18].U, c = 'w', edgecolors = 'k')
plt.scatter(datasets.iloc[18:21].u, datasets.iloc[18:21].U, c = mycolors[:3])
plt.scatter(datasets.iloc[21:].u, datasets.iloc[21:].U, c = mycolors[3:])

if 1:
    # Label each point
    for i in range(len(datasets)):
        y = datasets.iloc[i].U
        x = datasets.iloc[i].u
        label = str(datasets.iloc[i].series)
        plt.text(x, y, label, fontsize=10, ha='right', va='bottom')
    
plt.xlabel('$u$ (m/s)')
plt.ylabel('$H_s$ (K/m²)')
plt.ylim(0.05, 0.3)


## figure panel with R²

In [ ]:
### QUANTUM MODELS
dfs = []
for f in glob.glob("./QNN_results/R2s_QNN*.txt"):
    df = pd.read_csv(f, index_col=None).drop(columns = 'error').dropna()
    df["n_qubits"] = f[8]
    dfs.append(df)

results = pd.concat(dfs, ignore_index=True)

def parse_config(s):
    match = re.match(r"(\d+)qubits_circuit_([a-zA-Z]+)_([a-zA-Z]+)_freq(\d+)enc_(\d+)dec_([a-zA-Z]+)", s)
    if match:
        qubits, enc_angles, angles, n_enc, n_dec,  entanglement = match.groups()
        return int(qubits), int(n_enc), int(n_dec), entanglement, angles, enc_angles
    return None, None, None, None

results[['qubits', 'n_enc', 'n_dec', 'entanglement','angles','enc_angles']] = results['model'].apply(
    lambda x: pd.Series(parse_config(x))
)

results['nonlocal'] = (results.entanglement == 'nonlocal')*1.
results['n_params'] = results.qubits*(results.n_enc + results.n_dec)*results.angles.apply(lambda s: len(s)) + results.qubits*results.n_enc*results.enc_angles.apply(lambda s: len(s))


### classical validation R²
results_main = pd.read_csv('./QNN_results/QNN_r2.txt', header = 0)
results_main.columns =['key','model','r2_train_unscaled', 'r2_val_unscaled','r2_train', 'r2_val', 'mse_train', 'mse_val']


### merge
results = results.merge(
    results_main,
    on=["model", "key"],
)

cols = ["r2_train","r2_val","r2_train_unscaled","r2_val_unscaled","mse_train","mse_val","n_params"]
results[cols] = results[cols].apply(pd.to_numeric, errors="coerce")
results = results.rename(columns = {'series':'PALMrun', 'model':'model_name'})

dfqm = results[['n_enc', 'n_dec', 'local', 'r2', 'qubits', 'nonlocal', 'n_params',
       'r2_train', 'r2_val', 'r2_train_unscaled', 'r2_val_unscaled',
       'mse_train', 'mse_val', 'PALMrun']].groupby(['n_params','PALMrun']).aggregate(['mean','std'])

dfqm_byqubits = results[['n_enc', 'n_dec', 'local', 'r2', 'qubits', 'nonlocal', 'n_params',
       'r2_train', 'r2_val', 'r2_train_unscaled', 'r2_val_unscaled',
       'mse_train', 'mse_val', 'PALMrun']].groupby(['n_params','PALMrun','qubits']).aggregate(['mean','std'])

selfdf = results[results.PALMrun =='d1'].groupby(['model_name']).r2_val.aggregate(['mean','std','count']) #just pick any of the PALMruns to get unique
MODELS_QU = selfdf[selfdf['mean'] > 0.65].index.values # include the good performance band in later analysis

In [ ]:
results

In [ ]:
### CLASSICAL MODELS
r2s_cl = pd.read_csv('./cl_results/generalisation_r2s_cl.txt', index_col = 0)
r2s_cl.columns = ['model_name', 'PALMrun', 'r2','r2_unscaled']

r2s_cl_main = pd.read_csv('./cl_results/clNN_r2.txt', index_col = None, header = None)
r2s_cl_main.columns = ['runnr', 'model_name','r2_train_unscaled', 'r2_val_unscaled', 'r2_train', 'r2_val', 'mse_train', 'mse_val' ]

model_list_all = !ls './cl_models/'
model_list_2 = [s for s in model_list_all if (s[-2:] == '_1')]
models = []
n_params = []
model_list_1 = []
MODELS_CL = [] # list for comparison of vert. profiles
for model1 in model_list_all:
    model = tf.keras.models.load_model('./cl_models/' + model1)
    models += [model]
    n_params += [[model.name, model.count_params()]]
    model_list_1 += [model.name]
    if model.count_params() < 253: #252 params is the largest QNN
        MODELS_CL += [model.name]
r2s_cl = r2s_cl.merge(pd.DataFrame(n_params, columns = ['model_name', 'n_params'])).merge(r2s_cl_main)

dfm = r2s_cl.drop(columns = ['model_name']).groupby(['n_params','PALMrun']).aggregate(['mean','std'])

## calculation of the clNN generalisation R²

In [ ]:
jnp_train_inputs, jnp_val_inputs, jnp_train_outputs, jnp_val_outputs = dataloading.load_data(0)
jnp_gen_inputs, jnp_gen_outputs = dataloading.load_gen_data(0,series_to_load = [f"{s}{i}" for s in "d" for i in range(1, 4)] )
jnp_gen2_inputs, jnp_gen2_outputs = dataloading.load_gen_data(0, series_to_load = [f"{s}{i}" for s in "d" for i in range(4, 7)])

X_train_tensor = tf.convert_to_tensor(jnp_train_inputs, dtype=tf.float32)
X_test_tensor = tf.convert_to_tensor(jnp_val_inputs, dtype=tf.float32)
y_train_tensor = tf.convert_to_tensor(jnp_train_outputs, dtype=tf.float32)
y_test_tensor = tf.convert_to_tensor(jnp_val_outputs , dtype=tf.float32)

In [ ]:
len(models)*6

In [ ]:
recalculate = False
if recalculate:
    r2s_cl = []
    for i,my_model in enumerate(models):
            print(i)
            for ser in [f"{s}{i}" for s in "d" for i in range(1, 7)]:
                jnp_ser_inputs, jnp_ser_outputs = dataloading.load_gen_data(0, series_to_load = [ser], n_samples = 30000)
        
                y_hat =  my_model.predict(tf.convert_to_tensor(jnp_ser_inputs, dtype=tf.float32))
        
                y_hat_uz = dataloading.unpreprocess_target(y_hat)
                y_uz = dataloading.unpreprocess_target(jnp_ser_outputs)
                
                r2s_cl += [[my_model.name, ser, r2_score(jnp_ser_outputs, y_hat), r2_score( y_uz, y_hat_uz)]]
            pd.DataFrame(r2s_cl).to_csv('./cl_results/generalisation_r2s_cl.txt')
    print('Done.')

## calculation of the clNN vertical profiles

## calculation of the QNN generalisation R²

## QNN vertical profiles

## full figure

In [ ]:
with open("./cl_results/panel_b_data.pkl", "rb") as f:
    panel_b_data = pickle.load(f)
with open("./QNN_results/panel_b_data_qu.pkl", "rb") as f:
    panel_b_data_qu = pickle.load(f)

In [ ]:
panel_b_data_sel = [d for d in np.array(panel_b_data).flatten() if d.get("model") in MODELS_CL]
panel_b_data_qu_sel = [d for d in panel_b_data_qu if d.get("model") in MODELS_QU]

In [ ]:
fig = plt.figure(figsize=(6.77, 2.3))

gs = gridspec.GridSpec(
    1, 3, 
    width_ratios=[1, 1, 2],
    wspace=0.30
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
# panel c has a broken axis, need an extra grid
subgs = gridspec.GridSpecFromSubplotSpec(
    1, 3, width_ratios = [.7,4,1.3], subplot_spec=gs[0, 2], wspace=0.05
)

ax3_veryleft  = fig.add_subplot(subgs[0, 0])
ax3_left  = fig.add_subplot(subgs[0, 1], sharey=ax3_veryleft)
ax3_right = fig.add_subplot(subgs[0, 2], sharey=ax3_veryleft)

for ax, lab in zip([ax1, ax2], ["(a)", "(b)"]):
    ax.text(0.02, 0.96, lab, transform=ax.transAxes,
            va="top", ha="left", fontsize=9)

ax3_left.text(0.02, 0.96, "(c)", transform=ax3_veryleft.transAxes,
              va="top", ha="left", fontsize=9)

# panel a, datasets overview
ax1.scatter(datasets.iloc[:18].u, datasets.iloc[:18].U, c='w', edgecolor='k')
ax1.scatter(datasets.iloc[18:21].u, datasets.iloc[18:21].U,
            c=[mix_with_white(mycolors[0],0.5),
               mix_with_white(mycolors[1],0.5),
               mix_with_white(mycolors[2],0.5)],
            edgecolor=mycolors[:3])
ax1.scatter(datasets.iloc[21:].u, datasets.iloc[21:].U,
            c=[mix_with_white(mycolors[3],0.5),
               mix_with_white(mycolors[4],0.5),
               mix_with_white(mycolors[5],0.5)],
            edgecolor=mycolors[3:])

ax1.set_xlabel('Geostrophic wind $u_g$ (m/s)')
ax1.set_ylabel('Surface heat flux $H_s$ (K/m²)')
ax1.set_ylim(0.05, 0.3)

# panel b, vertical profiles
def aggregate(panel_data, key):
    grouped = defaultdict(list)
    for d in panel_data:
        grouped[d["series"]].append(d)

    out = []
    for ser, items in grouped.items():
        stack = np.stack([d[key] for d in items])
        out.append(dict(
            series=ser,
            z=items[0]["z"],
            mean=np.nanmean(stack, axis=0),
            std=np.nanstd(stack, axis=0),
        ))
    return out

avg_cl = aggregate(panel_b_data_sel, "mse_classical")
avg_qu = aggregate(panel_b_data_qu_sel, "mse_quantum")

for i, d in enumerate(avg_cl):
    lower = np.clip(d["mean"] - d["std"], 1e-12, None)
    upper = d["mean"] + d["std"]
    ax2.plot(d["mean"][~np.isnan(d['mean'])], d["z"][~np.isnan(d['mean'])], linestyle="--",
             color=mycolors[i], linewidth=1.)
    ax2.fill_betweenx(d["z"][~np.isnan(d['mean'])], lower[~np.isnan(d['mean'])], upper[~np.isnan(d['mean'])],
                      color=mycolors[i], alpha=0.2, linewidth=0)

for i, d in enumerate(avg_qu):
    lower = np.clip(d["mean"] - d["std"], 1e-12, None)
    upper = d["mean"] + d["std"]
    ax2.plot(d["mean"][~np.isnan(d['mean'])], d["z"][~np.isnan(d['mean'])], linestyle="-",
             color=mycolors[i], linewidth=1.)
    ax2.fill_betweenx(d["z"][~np.isnan(d['mean'])], lower[~np.isnan(d['mean'])], upper[~np.isnan(d['mean'])],
                      color=mycolors[i], alpha=0.2, linewidth=0)

ax2.set_xlabel("MSE")
ax2.set_ylabel(r"$z/z_{\rm ABL}$")
ax2.set_ylim(0.2, 0.8)
ax2.set_xscale("log")
ax2.set_xlim(0.01, 1)

# panel c with the R²
j = 0
for ser, group in dfqm_byqubits.groupby('PALMrun'):
    for ax in [ax3_veryleft, ax3_left, ax3_right]:
        ax.errorbar(group[('r2_val','mean')],
                    group.r2['mean'],
                    group.r2['std'],
                    linestyle='',
                    marker='o',
                    color=mycolors[j],
                    markerfacecolor=mix_with_white(mycolors[j]),
                    markersize=3.3)
    j += 1

j = 0
for ser, group in dfm.groupby('PALMrun'):
    for ax in [ax3_veryleft, ax3_left, ax3_right]:
        ax.errorbar(group[('r2_val','mean')],
                    group.r2['mean'],
                    group.r2['std'],
                    linestyle='',
                    marker='s',
                    color=mycolors[j],
                    markerfacecolor=mix_with_white(mycolors[j]),
                    markeredgecolor='k',
                    markersize=3.3)
    j += 1

# layout
ax3_veryleft.set_xlim(0.55, 0.62)
ax3_left.set_xlim(0.64, 0.74)
ax3_right.set_xlim(0.75, 0.84)
ax3_left.set_ylim(0.35, 0.85)

ax3_veryleft.set_ylabel('generalisation $R^2$')
ax3_left.set_xlabel('validation $R^2$')
ax3_right.set_xlabel('')

ax3_veryleft.spines['right'].set_visible(False)
ax3_left.spines['left'].set_visible(False)
ax3_left.spines['right'].set_visible(False)
ax3_right.spines['left'].set_visible(False)

ax3_left.yaxis.tick_right()
ax3_left.tick_params(labelright=False)
ax3_right.yaxis.tick_right()
ax3_right.tick_params(labelright=False)
ax3_right.tick_params(left = False)
ax3_left.tick_params(right = False)

d = .015
kwargs = dict(transform=ax3_left.transAxes, color='k', clip_on=False)
ax3_left.plot((1-d, 1+d), (-d, +d), **kwargs)
ax3_left.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

kwargs.update(transform=ax3_right.transAxes)
ax3_right.plot((-d, +d), (-d, +d), **kwargs)
ax3_right.plot((-d, +d), (1-d, 1+d), **kwargs)

d = .015
kwargs = dict(transform=ax3_veryleft.transAxes, color='k', clip_on=False)
ax3_veryleft.plot((1-d, 1+d), (-d, +d), **kwargs)
ax3_veryleft.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

kwargs.update(transform=ax3_left.transAxes)
ax3_left.plot((-d, +d), (-d, +d), **kwargs)
ax3_left.plot((-d, +d), (1-d, 1+d), **kwargs)

plt.subplots_adjust(
    left=0.08,
    right=0.90,
    bottom=0.22,
    top=0.90
)

fig.align_xlabels()

fig.savefig("./figures/generalisation_3panel_broken.pdf")

In [ ]:
dfm